# RoadSafe Chicago | Partie 01 : Collecte des données

## Le projet

**RoadSafe Chicago** est une étude de l'accidentologie routière à Chicago, à partir des données ouvertes de la ville.

**Problématique** : *Comment évoluent la fréquence et la gravité des accidents routiers à Chicago, et quels facteurs sont associés à leur gravité ?*

L'analyse s'organise autour de quatre axes :
- **Quand ?** Saisonnalité, jours, heures, évolution annuelle.
- **Où ?** Répartition géographique des accidents.
- **Dans quel contexte ?** Météo, luminosité, type de route, vitesse.
- **Pour qui ?** Usagers, véhicules impliqués, gravité des blessures.

Le projet suit quatre grandes étapes : **collecter → fiabiliser → analyser → restituer**.

---

## Objectif de cette partie

Récupérer de manière **fiable et reproductible** l'ensemble des accidents enregistrés à Chicago en **2024**, puis les sauvegarder dans un fichier brut vérifié.

## Source

- **City of Chicago Open Data**, jeu de données *Traffic Crashes – Crashes* (`85ca-t3if`)
- Accès via l'**API Socrata**, au format JSON
- **1 ligne = 1 accident**, identifié par `crash_record_id`

## Contenu de cette partie

1. **Volume attendu** : interrogation de l'API pour connaître le nombre d'accidents en 2024.
2. **Extraction paginée** : téléchargement des données par blocs de 10 000 lignes, avec gestion des erreurs et des délais du serveur.
3. **Contrôles** : comparaison du volume extrait avec celui de l'API, vérification de l'unicité des accidents et de la validité des dates.
4. **Sauvegarde** : écriture du fichier brut `data/raw/crashes_2024.parquet`, accompagné d'un fichier de métadonnées (date de collecte, volumes).

## Résultat attendu

Un jeu de données brut, complet et traçable, prêt pour la partie suivante : **la compréhension des données** (*Data Understanding*).

## Librairies 

In [2]:
import pandas as pd 
import requests 
import os 
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry



## 1. Importation des données 

In [3]:
URL = "https://data.cityofchicago.org/resource/85ca-t3if.json"

year = 2024
limit = 10000
offset = 0

token = os.getenv("SOCRATA_APP_TOKEN")
headers = {"X-App-Token": token} if token else {}


session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=5, backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])))
session.headers.update(headers)



where = f"crash_date >= '{year}-01-01T00:00:00' AND crash_date < '{year+1}-01-01T00:00:00'"


In [4]:
# 1. Volume attendu
r = session.get(URL, params={"$select": "count(*)", "$where": where}, timeout=(10, 60))
r.raise_for_status()
expected = int(r.json()[0]["count"])
print("Lignes attendues :", expected)

Lignes attendues : 112056


**Analyse.** L'API annonce **112 056 accidents** pour 2024, soit environ 306 par jour. Ce chiffre sert de référence pour valider l'extraction : le volume téléchargé devra lui être strictement égal.

## 2. Extraction paginée

In [5]:

chunks = []
while True:
    params = {"$where": where, "$limit": limit, "$offset": offset, "$order": ":id"}  # tri rapide
    r = session.get(URL, params=params, timeout=(10, 180))                          # timeout plus long
    r.raise_for_status()
    batch = r.json()
    if not batch:
        break
    chunks.append(pd.DataFrame(batch))
    offset += limit
    print(f"{offset} / {expected} lignes récupérées")

10000 / 112056 lignes récupérées
20000 / 112056 lignes récupérées
30000 / 112056 lignes récupérées
40000 / 112056 lignes récupérées
50000 / 112056 lignes récupérées
60000 / 112056 lignes récupérées
70000 / 112056 lignes récupérées
80000 / 112056 lignes récupérées
90000 / 112056 lignes récupérées
100000 / 112056 lignes récupérées
110000 / 112056 lignes récupérées
120000 / 112056 lignes récupérées


**Analyse.** Le compteur affiche 120 000 au dernier tour car il avance par pas de 10 000 : le dernier bloc ne contient en réalité que 2 056 lignes. Le tri explicite sur `:id` est indispensable, car sans ordre fixe l'API Socrata peut renvoyer des lignes en double ou en sauter d'une page à l'autre.

In [6]:

df = pd.concat(chunks, ignore_index=True)


## 3. Contrôles + sauvegarde

In [7]:
# Affichage des informations sur le DataFrame
display(df.head(3))

,crash_record_id,crash_date,posted_speed_limit,traffic_control_device,device_condition,weather_condition,lighting_condition,first_crash_type,trafficway_type,alignment,...,location,intersection_related_i,crash_date_est_i,photos_taken_i,statements_taken_i,work_zone_i,work_zone_type,workers_present_i,dooring_i,lane_cnt
0,5082d6e9f9c5c96660e7ec7fd416549e50d3130af3fc5c...,2024-06-27T13:45:00.000,10,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PARKED MOTOR VEHICLE,PARKING LOT,STRAIGHT AND LEVEL,...,"{'type': 'Point', 'coordinates': ['-87.683127'...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fd34aa4f12f00106b32134531fc6e805590dbe89fef502...,2024-06-22T18:07:00.000,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PARKED MOTOR VEHICLE,NOT DIVIDED,STRAIGHT AND LEVEL,...,"{'type': 'Point', 'coordinates': ['-87.646149'...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,f0a652bf48a8c08c8a26cc981e39ed757cd41f8a771761...,2024-06-23T08:41:00.000,10,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,REAR TO REAR,PARKING LOT,STRAIGHT AND LEVEL,...,"{'type': 'Point', 'coordinates': ['-87.652629'...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
print ( "information sur le DataFrame :")
df.info()

print ( "Nombre de valeurs manquantes par colonne :")
df.isnull().sum()




information sur le DataFrame :
<class 'pandas.DataFrame'>
RangeIndex: 112056 entries, 0 to 112055
Data columns (total 49 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   crash_record_id                112056 non-null  str   
 1   crash_date                     112056 non-null  str   
 2   posted_speed_limit             112056 non-null  str   
 3   traffic_control_device         112056 non-null  str   
 4   device_condition               112056 non-null  str   
 5   weather_condition              112056 non-null  str   
 6   lighting_condition             112056 non-null  str   
 7   first_crash_type               112056 non-null  str   
 8   trafficway_type                112056 non-null  str   
 9   alignment                      112056 non-null  str   
 10  roadway_surface_cond           112056 non-null  str   
 11  road_defect                    112056 non-null  str   
 12  report_type             

crash_record_id                       0
crash_date                            0
posted_speed_limit                    0
traffic_control_device                0
device_condition                      0
weather_condition                     0
lighting_condition                    0
first_crash_type                      0
trafficway_type                       0
alignment                             0
roadway_surface_cond                  0
road_defect                           0
report_type                        5071
crash_type                            0
private_property_i               107506
hit_and_run_i                     76183
damage                                0
date_police_notified                  0
prim_contributory_cause               0
sec_contributory_cause                0
street_no                             0
street_direction                      0
street_name                           0
beat_of_occurrence                    0
num_units                             0


**Analyse.** Les 49 colonnes arrivent toutes au format texte (c'est le format JSON de l'API), d'où l'étape de typage prévue en partie 03. Les valeurs manquantes se concentrent sur des indicateurs optionnels (`private_property_i`, `hit_and_run_i`, zones de travaux...) : pour ces colonnes, l'absence de valeur signifie le plus souvent « non » ou « non concerné » plutôt qu'une information perdue. Les variables centrales pour l'étude (date, conditions, type de collision) sont complètes.

In [9]:
# Verification des valeurs dupliquées

print( " Verification des valeurs dupliquées :")

df["crash_record_id"].duplicated().sum()


 Verification des valeurs dupliqués :


np.int64(0)

**Analyse.** Aucun doublon sur `crash_record_id` et un volume extrait identique au volume annoncé par l'API : l'extraction est **complète** et la clé est **fiable**. Le fichier brut peut être sauvegardé tel quel, sans aucune transformation.

In [11]:
# Sauvegarde du DataFrame en format Parquet 
os.makedirs("../data/raw", exist_ok=True)
df.to_parquet(f"../data/raw/crashes_{year}.parquet", index=False)

## Bilan

- **112 056 accidents** récupérés pour 2024, conformes au volume annoncé par l'API.
- Clé `crash_record_id` unique, aucune ligne perdue ni dupliquée.
- Toutes les colonnes sont en texte : le typage et le contrôle des valeurs sont traités en parties 02 et 03.
- Le fichier brut `data/raw/crashes_2024.parquet` n'est plus jamais modifié par la suite.